In [ ]:
!pip -q install gradio

import re
import warnings
import html as _html

import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

# ---------- 1. Upload dataset ----------
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload a CSV dataset.")

df = pd.read_csv(next(iter(uploaded)))

if "fraudulent" not in df.columns:
    raise ValueError("The CSV must contain a 'fraudulent' column.")

df["fraudulent"] = pd.to_numeric(df["fraudulent"], errors="coerce")
df = df.dropna(subset=["fraudulent"]).copy()
df["fraudulent"] = df["fraudulent"].astype(int)

if df["fraudulent"].nunique() != 2:
    raise ValueError(
        "The 'fraudulent' column must contain both classes: 0 and 1."
    )

# ---------- 2. Dataset summary ----------
print("Dataset shape:", df.shape)
print("\nClass balance:\n", df["fraudulent"].value_counts())
print(
    "\nMissing values (%):\n",
    (df.isna().mean() * 100)
    .round(1)
    .sort_values(ascending=False)
    .head(10),
)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
df["fraudulent"].value_counts().sort_index().plot.bar(
    ax=ax[0],
    title="Genuine (0) vs Fraudulent (1)",
)

if "employment_type" in df.columns:
    df.groupby("employment_type", dropna=False)["fraudulent"].mean().plot.bar(
        ax=ax[1],
        title="Fraud rate by employment type",
    )
else:
    ax[1].set_title("employment_type column unavailable")

if "has_company_logo" in df.columns:
    df.groupby("has_company_logo", dropna=False)["fraudulent"].mean().plot.bar(
        ax=ax[2],
        title="Fraud rate by company logo",
    )
else:
    ax[2].set_title("has_company_logo column unavailable")

plt.tight_layout()
plt.show()

# ---------- 3. Preprocessing ----------
TEXT_COLS = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
]
CAT_COLS = [
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
]
BINARY_COLS = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
]
NUM_COLS = BINARY_COLS + [
    "text_len",
    "caps_ratio",
    "has_salary",
    "has_profile",
    "n_links",
    "n_exclaim",
]
ALL_INPUTS = TEXT_COLS + CAT_COLS + [
    "location",
    "salary_range",
] + BINARY_COLS


def clean_text(value):
    value = str(value).lower()
    value = re.sub(r"http\S+|www\.\S+", " url ", value)
    value = re.sub(r"\S+@\S+", " email ", value)
    value = re.sub(r"[^a-z\s]", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def add_features(dataframe):
    data = dataframe.copy()

    for column in ALL_INPUTS:
        if column not in data.columns:
            data[column] = 0 if column in BINARY_COLS else ""

    for column in TEXT_COLS + CAT_COLS + ["location", "salary_range"]:
        data[column] = data[column].fillna("").astype(str)

    for column in BINARY_COLS:
        data[column] = (
            pd.to_numeric(data[column], errors="coerce")
            .fillna(0)
            .astype(int)
        )

    for column in CAT_COLS:
        data[column] = data[column].replace("", "unknown")

    raw_text = data[TEXT_COLS].agg(" ".join, axis=1)
    data["text"] = raw_text.map(clean_text)
    data["text_len"] = raw_text.str.len()
    data["caps_ratio"] = raw_text.map(
        lambda value: sum(char.isupper() for char in value)
        / max(len(value), 1)
    )
    data["has_salary"] = (
        data["salary_range"].str.strip() != ""
    ).astype(int)
    data["has_profile"] = (
        data["company_profile"].str.strip() != ""
    ).astype(int)
    data["n_links"] = raw_text.str.count(r"http|www\.|@")
    data["n_exclaim"] = raw_text.str.count("!")

    return data


data = add_features(df)
X = data.drop(columns=["fraudulent"])
y = data["fraudulent"]

from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("\nTraining rows:", len(Xtr))
print("Testing rows:", len(Xte))

# ---------- 4. Preprocessing and models ----------
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)


def make_one_hot_encoder():
    # Works with both older and newer scikit-learn versions.
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
        )


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "text",
                TfidfVectorizer(
                    max_features=5000,
                    stop_words="english",
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "text",
            ),
            ("cat", make_one_hot_encoder(), CAT_COLS),
            ("num", MinMaxScaler(), NUM_COLS),
        ],
        remainder="drop",
    )


models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=30,
        class_weight="balanced",
        random_state=42,
    ),
    "Naive Bayes": MultinomialNB(alpha=0.1),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
    ),
    "SVM": CalibratedClassifierCV(
        LinearSVC(class_weight="balanced"),
        cv=3,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    ),
}

# ---------- 5. Train and compare models ----------
rows = []
fitted_models = {}

for setting in ["Without feature selection", "With chi-square selection"]:
    for model_name, classifier in models.items():
        prep = make_preprocessor()
        Xtr_prepared = prep.fit_transform(Xtr, ytr)

        if setting == "With chi-square selection":
            # Select at most 2,000 features; this avoids errors on smaller datasets.
            k = min(2000, Xtr_prepared.shape[1])
            selector = SelectKBest(score_func=chi2, k=k)
            Xtr_model = selector.fit_transform(Xtr_prepared, ytr)
        else:
            selector = None
            Xtr_model = Xtr_prepared

        model = clone(classifier)
        model.fit(Xtr_model, ytr)

        Xte_prepared = prep.transform(Xte)
        Xte_model = (
            selector.transform(Xte_prepared)
            if selector is not None
            else Xte_prepared
        )

        predictions = model.predict(Xte_model)
        probabilities = model.predict_proba(Xte_model)[:, 1]

        steps = [("prep", prep)]
        if selector is not None:
            steps.append(("feature_selection", selector))
        steps.append(("classifier", model))
        pipeline = Pipeline(steps)

        scores = {
            "Setting": setting,
            "Model": model_name,
            "Accuracy": accuracy_score(yte, predictions),
            "Precision": precision_score(
                yte, predictions, zero_division=0
            ),
            "Recall": recall_score(
                yte, predictions, zero_division=0
            ),
            "F1": f1_score(yte, predictions, zero_division=0),
            "ROC-AUC": roc_auc_score(yte, probabilities),
        }

        rows.append(scores)
        fitted_models[(setting, model_name)] = pipeline

        print(
            f"{setting:28s} {model_name:20s} "
            f"F1={scores['F1']:.3f}"
        )

# ---------- 6. Show evaluation results ----------
results = pd.DataFrame(rows).round(4)
display(results)

results.pivot(
    index="Model",
    columns="Setting",
    values="F1",
).plot.bar(
    figsize=(10, 5),
    title="Model comparison by F1 score",
)
plt.ylabel("F1 score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

best = results.sort_values(
    ["F1", "ROC-AUC"],
    ascending=False,
).iloc[0]

best_key = (best["Setting"], best["Model"])
best_model = fitted_models[best_key]

print(
    f"\nBest model: {best['Model']} | {best['Setting']} | "
    f"F1={best['F1']} | Recall={best['Recall']}"
)
print(
    "Confusion matrix:\n",
    confusion_matrix(yte, best_model.predict(Xte)),
)

# ---------- 7. Global word explanation ----------
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=2,
)
Xtr_text = tfidf.fit_transform(Xtr["text"])

word_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
)
word_model.fit(Xtr_text, ytr)

word_weights = pd.Series(
    word_model.coef_[0],
    index=tfidf.get_feature_names_out(),
).sort_values()

top_words = pd.concat(
    [word_weights.head(15), word_weights.tail(15)]
)
bar_colors = ["tab:green"] * 15 + ["tab:red"] * 15

top_words.plot.barh(
    figsize=(8, 7),
    color=bar_colors,
    title="Word weights (green = genuine, red = fraudulent)",
)
plt.tight_layout()
plt.show()

# ---------- 8. Prediction and explanations ----------
def probability_fake(row_dict):
    row = add_features(pd.DataFrame([row_dict]))
    return float(best_model.predict_proba(row)[0, 1])


def explain_words(form, base_probability, top_n=6, max_words=60):
    combined_text = " ".join(
        str(form.get(column, ""))
        for column in TEXT_COLS
    )

    words = list(
        dict.fromkeys(re.findall(r"[A-Za-z]{4,}", combined_text.lower()))
    )[:max_words]

    changes = []

    for word in words:
        modified = {
            key: (
                re.sub(
                    rf"\b{re.escape(word)}\b",
                    " ",
                    str(value),
                    flags=re.I,
                )
                if key in TEXT_COLS
                else value
            )
            for key, value in form.items()
        }

        change = base_probability - probability_fake(modified)
        changes.append((word, change))

    changes.sort(key=lambda item: item[1])

    fake_words = [
        (word, round(change * 100, 2))
        for word, change in changes[::-1][:top_n]
        if change > 0
    ]
    genuine_words = [
        (word, round(change * 100, 2))
        for word, change in changes[:top_n]
        if change < 0
    ]

    return fake_words, genuine_words


# ---------- 9. Gradio interface ----------
CSS = """
.gradio-container {
    background:
      radial-gradient(ellipse at top left, #dbeafe 0%, transparent 45%),
      linear-gradient(135deg, #eff6ff 0%, #f5f3ff 50%, #fdf2f8 100%);
    min-height: 100vh;
}

.fj-hero {
    background: linear-gradient(120deg, #1e3a8a, #6d28d9 60%, #db2777);
    color: white;
    padding: 26px 30px;
    border-radius: 16px;
    margin-bottom: 12px;
}

.fj-hero h1 {
    margin: 0 0 6px;
    font-size: 30px;
    color: white;
}

.fj-hero p {
    margin: 0;
    opacity: .92;
    font-size: 15px;
}

.fj-pills span {
    display: inline-block;
    background: rgba(255,255,255,.18);
    padding: 4px 12px;
    border-radius: 20px;
    margin: 10px 6px 0 0;
    font-size: 13px;
}

.fj-card {
    background: white;
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 22px;
    color: #111827;
}

.fj-top {
    display: flex;
    gap: 20px;
    align-items: center;
    flex-wrap: wrap;
}

.fj-gauge {
    width: 130px;
    height: 130px;
    border-radius: 50%;
    display: grid;
    place-items: center;
    background: conic-gradient(
        var(--c) calc(var(--p) * 1%),
        #e5e7eb 0
    );
}

.fj-gauge-inner {
    width: 104px;
    height: 104px;
    border-radius: 50%;
    background: white;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
}

.fj-gauge b {
    font-size: 25px;
}

.fj-gauge small {
    font-size: 11px;
    color: #6b7280;
}

.fj-badge {
    color: white;
    padding: 7px 12px;
    border-radius: 20px;
}

.fj-flag {
    background: #fff7ed;
    border-left: 4px solid #f97316;
    padding: 8px 12px;
    margin: 6px 0;
    border-radius: 6px;
    font-size: 14px;
    color: #7c2d12;
}

.fj-ok {
    background: #ecfdf5;
    border-left: 4px solid #10b981;
    padding: 8px 12px;
    border-radius: 6px;
    font-size: 14px;
    color: #065f46;
}

.fj-chip {
    display: inline-block;
    padding: 4px 11px;
    margin: 3px 4px 3px 0;
    border-radius: 14px;
    font-size: 13px;
}

.fj-red {
    background: #fee2e2;
    color: #991b1b;
}

.fj-green {
    background: #dcfce7;
    color: #166534;
}

.fj-tip {
    margin-top: 14px;
    font-size: 12.5px;
    color: #6b7280;
}

.fj-foot {
    text-align: center;
    color: #6b7280;
    font-size: 13px;
    margin-top: 8px;
}

.fj-foot span {
    display: inline-block;
    background: #eef2ff;
    color: #3730a3;
    padding: 3px 10px;
    border-radius: 12px;
    margin: 3px;
}
"""

HERO = f"""
<div class="fj-hero">
  <h1>🛡️ Fake Job &amp; Internship Detector</h1>
  <p>Enter a job posting to check for possible scam patterns.</p>
  <div class="fj-pills">
    <span>🤖 {_html.escape(str(best['Model']))}</span>
    <span>🎯 F1: {best['F1']}</span>
    <span>🔎 Explainable AI</span>
  </div>
</div>
"""

PLACEHOLDER = """
<div class="fj-card" style="text-align:center;padding:40px 20px">
  <div style="font-size:54px">🕵️</div>
  <h3>Waiting for a posting</h3>
  <p style="color:#6b7280">
    Fill in the form, then press Analyze posting.
  </p>
</div>
"""


def render_result(probability, fake_words, genuine_words, flags):
    percentage = round(probability * 100, 1)

    if probability >= 0.5:
        color = "#ef4444"
        verdict = "⚠️ FAKE / SUSPICIOUS"
        risk = "High risk"
    elif probability >= 0.3:
        color = "#f59e0b"
        verdict = "🤔 NEEDS CAUTION"
        risk = "Medium risk"
    else:
        color = "#10b981"
        verdict = "✅ LIKELY GENUINE"
        risk = "Low risk"

    flags_html = "".join(
        f'<div class="fj-flag">🚩 {_html.escape(flag)}</div>'
        for flag in flags
    )
    if not flags_html:
        flags_html = (
            '<div class="fj-ok">'
            "✔ No rule-based red flags found"
            "</div>"
        )

    fake_html = "".join(
        f'<span class="fj-chip fj-red">'
        f'{_html.escape(word)} +{change}%</span>'
        for word, change in fake_words
    )
    if not fake_html:
        fake_html = '<span style="color:#6b7280">None identified</span>'

    genuine_html = "".join(
        f'<span class="fj-chip fj-green">'
        f'{_html.escape(word)} {change}%</span>'
        for word, change in genuine_words
    )
    if not genuine_html:
        genuine_html = (
            '<span style="color:#6b7280">None identified</span>'
        )

    return f"""
    <div class="fj-card">
      <div class="fj-top">
        <div class="fj-gauge" style="--p:{percentage};--c:{color}">
          <div class="fj-gauge-inner">
            <b>{percentage}%</b>
            <small>fake probability</small>
          </div>
        </div>
        <div>
          <span class="fj-badge" style="background:{color}">
            {verdict}
          </span>
          <p style="margin:10px 0 0;color:#4b5563">
            {risk}. Verify the company through its official website
            before sharing documents or paying fees.
          </p>
        </div>
      </div>
      <h3>🚩 Red flags</h3>
      {flags_html}
      <h3>🔴 Words that increased the fake score</h3>
      {fake_html}
      <h3>🟢 Words that lowered the fake score</h3>
      {genuine_html}
      <div class="fj-tip">
        The word explanation removes individual words and measures how
        the model's fake probability changes.
      </div>
    </div>
    """


def analyze(
    title,
    location,
    salary,
    industry,
    function,
    employment_type,
    experience,
    education,
    profile,
    description,
    requirements,
    benefits,
    remote,
    logo,
    questions,
):
    form = {
        "title": title or "",
        "location": location or "",
        "salary_range": salary or "",
        "industry": industry or "",
        "function": function or "",
        "employment_type": employment_type or "",
        "required_experience": experience or "",
        "required_education": education or "",
        "company_profile": profile or "",
        "description": description or "",
        "requirements": requirements or "",
        "benefits": benefits or "",
        "telecommuting": int(bool(remote)),
        "has_company_logo": int(bool(logo)),
        "has_questions": int(bool(questions)),
    }

    if not form["title"].strip() and not form["description"].strip():
        return PLACEHOLDER

    probability = probability_fake(form)
    fake_words, genuine_words = explain_words(form, probability)

    all_text = " ".join(str(value) for value in form.values())
    flags = []

    if not form["company_profile"].strip():
        flags.append("No company profile was provided")

    if not form["has_company_logo"]:
        flags.append("No company logo was indicated")

    if re.search(r"@(gmail|yahoo|outlook|hotmail)\.", all_text, re.I):
        flags.append("A free email address appears in the posting")

    if re.search(
        r"registration fee|deposit|western union|whatsapp|pay ",
        all_text,
        re.I,
    ):
        flags.append("The posting mentions payment or messaging apps")

    if re.search(
        r"no experience|urgent|earn \w+ (daily|weekly)|guaranteed",
        all_text,
        re.I,
    ):
        flags.append("The posting uses urgent or unusually attractive wording")

    return render_result(
        probability,
        fake_words,
        genuine_words,
        flags,
    )


# ---------- 10. Build the interface ----------
EMPLOYMENT_TYPES = [
    "",
    "Full-time",
    "Part-time",
    "Contract",
    "Temporary",
    "Other",
]
EXPERIENCE_LEVELS = [
    "",
    "Internship",
    "Entry level",
    "Associate",
    "Mid-Senior level",
    "Director",
    "Executive",
    "Not Applicable",
]
EDUCATION_LEVELS = [
    "",
    "Bachelor's Degree",
    "Master's Degree",
    "High School or equivalent",
    "Certification",
    "Unspecified",
]

with gr.Blocks(
    title="Fake Job & Internship Detector",
    css=CSS,
) as demo:
    gr.HTML(HERO)

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### 📝 Job details")

            with gr.Row():
                title_input = gr.Textbox(
                    label="Job title",
                    placeholder="e.g. Python Developer Intern",
                )
                location_input = gr.Textbox(
                    label="Location",
                    placeholder="e.g. IN, Hyderabad",
                )

            with gr.Row():
                salary_input = gr.Textbox(
                    label="Salary / stipend",
                    placeholder="e.g. 10000-15000 per month",
                )
                industry_input = gr.Textbox(
                    label="Industry",
                    placeholder="Information Technology",
                )
                function_input = gr.Textbox(
                    label="Function",
                    placeholder="Engineering",
                )

            with gr.Row():
                employment_input = gr.Dropdown(
                    choices=EMPLOYMENT_TYPES,
                    value="",
                    label="Employment type",
                )
                experience_input = gr.Dropdown(
                    choices=EXPERIENCE_LEVELS,
                    value="",
                    label="Experience level",
                )
                education_input = gr.Dropdown(
                    choices=EDUCATION_LEVELS,
                    value="",
                    label="Education",
                )

            profile_input = gr.Textbox(
                label="Company profile",
                lines=2,
                placeholder="About the company...",
            )

            description_input = gr.Textbox(
                label="Job description",
                lines=5,
                placeholder="Paste the job posting here...",
            )

            with gr.Row():
                requirements_input = gr.Textbox(
                    label="Requirements",
                    lines=2,
                )
                benefits_input = gr.Textbox(
                    label="Benefits",
                    lines=2,
                )

            with gr.Row():
                remote_input = gr.Checkbox(label="🏠 Remote job")
                logo_input = gr.Checkbox(
                    label="🏢 Company logo present",
                    value=True,
                )
                questions_input = gr.Checkbox(
                    label="❓ Screening questions",
                )

            with gr.Row():
                analyze_button = gr.Button(
                    "🔍 Analyze posting",
                    variant="primary",
                )
                clear_button = gr.ClearButton(value="Clear")

        with gr.Column(scale=4):
            output = gr.HTML(value=PLACEHOLDER)

    input_components = [
        title_input,
        location_input,
        salary_input,
        industry_input,
        function_input,
        employment_input,
        experience_input,
        education_input,
        profile_input,
        description_input,
        requirements_input,
        benefits_input,
        remote_input,
        logo_input,
        questions_input,
    ]

    analyze_button.click(
        fn=analyze,
        inputs=input_components,
        outputs=output,
    )
    clear_button.add(input_components + [output])

    gr.HTML(
        """
        <div class="fj-foot">
          <span>🎯 SDG 8: Decent Work &amp; Economic Growth</span>
          <span>📚 SDG 4: Quality Education</span>
          <span>⚖️ SDG 16: Strong Institutions</span>
        </div>
        """
    )

# share=True creates a public Gradio link. Remove it for a local-only app.
demo.launch(share=True)

In [ ]:
!pip -q install gradio

import re
import warnings
import html as _html

import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

# ---------- 1. Upload dataset ----------
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload a CSV dataset.")

df = pd.read_csv(next(iter(uploaded)))

if "fraudulent" not in df.columns:
    raise ValueError("The CSV must contain a 'fraudulent' column.")

df["fraudulent"] = pd.to_numeric(df["fraudulent"], errors="coerce")
df = df.dropna(subset=["fraudulent"]).copy()
df["fraudulent"] = df["fraudulent"].astype(int)

if df["fraudulent"].nunique() != 2:
    raise ValueError(
        "The 'fraudulent' column must contain both classes: 0 and 1."
    )

# ---------- 2. Dataset summary ----------
print("Dataset shape:", df.shape)
print("\nClass balance:\n", df["fraudulent"].value_counts())
print(
    "\nMissing values (%):\n",
    (df.isna().mean() * 100)
    .round(1)
    .sort_values(ascending=False)
    .head(10),
)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
df["fraudulent"].value_counts().sort_index().plot.bar(
    ax=ax[0],
    title="Genuine (0) vs Fraudulent (1)",
)

if "employment_type" in df.columns:
    df.groupby("employment_type", dropna=False)["fraudulent"].mean().plot.bar(
        ax=ax[1],
        title="Fraud rate by employment type",
    )
else:
    ax[1].set_title("employment_type column unavailable")

if "has_company_logo" in df.columns:
    df.groupby("has_company_logo", dropna=False)["fraudulent"].mean().plot.bar(
        ax=ax[2],
        title="Fraud rate by company logo",
    )
else:
    ax[2].set_title("has_company_logo column unavailable")

plt.tight_layout()
plt.show()

# ---------- 3. Preprocessing ----------
TEXT_COLS = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
]

CAT_COLS = [
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
]

BINARY_COLS = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
]

NUM_COLS = BINARY_COLS + [
    "text_len",
    "caps_ratio",
    "has_salary",
    "has_profile",
    "n_links",
    "n_exclaim",
]

ALL_INPUTS = TEXT_COLS + CAT_COLS + [
    "location",
    "salary_range",
] + BINARY_COLS


def clean_text(value):
    value = str(value).lower()
    value = re.sub(r"http\S+|www\.\S+", " url ", value)
    value = re.sub(r"\S+@\S+", " email ", value)
    value = re.sub(r"[^a-z\s]", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def add_features(dataframe):
    data = dataframe.copy()

    for column in ALL_INPUTS:
        if column not in data.columns:
            data[column] = 0 if column in BINARY_COLS else ""

    for column in TEXT_COLS + CAT_COLS + ["location", "salary_range"]:
        data[column] = data[column].fillna("").astype(str)

    for column in BINARY_COLS:
        data[column] = (
            pd.to_numeric(data[column], errors="coerce")
            .fillna(0)
            .astype(int)
        )

    for column in CAT_COLS:
        data[column] = data[column].replace("", "unknown")

    raw_text = data[TEXT_COLS].agg(" ".join, axis=1)
    data["text"] = raw_text.map(clean_text)
    data["text_len"] = raw_text.str.len()
    data["caps_ratio"] = raw_text.map(
        lambda value: sum(char.isupper() for char in value)
        / max(len(value), 1)
    )
    data["has_salary"] = (
        data["salary_range"].str.strip() != ""
    ).astype(int)
    data["has_profile"] = (
        data["company_profile"].str.strip() != ""
    ).astype(int)
    data["n_links"] = raw_text.str.count(r"http|www\.|@")
    data["n_exclaim"] = raw_text.str.count("!")

    return data


data = add_features(df)
X = data.drop(columns=["fraudulent"])
y = data["fraudulent"]

from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("\nTraining rows:", len(Xtr))
print("Testing rows:", len(Xte))

# ---------- 4. Preprocessing and models ----------
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)


def make_one_hot_encoder():
    # Compatible with older and newer scikit-learn versions.
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
        )


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "text",
                TfidfVectorizer(
                    max_features=5000,
                    stop_words="english",
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "text",
            ),
            ("cat", make_one_hot_encoder(), CAT_COLS),
            ("num", MinMaxScaler(), NUM_COLS),
        ],
        remainder="drop",
    )


models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=30,
        class_weight="balanced",
        random_state=42,
    ),
    "Naive Bayes": MultinomialNB(alpha=0.1),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
    ),
    "SVM": CalibratedClassifierCV(
        LinearSVC(class_weight="balanced"),
        cv=3,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    ),
}

# ---------- 5. Train and compare models ----------
rows = []
fitted_models = {}

for setting in ["Without feature selection", "With chi-square selection"]:
    for model_name, classifier in models.items():
        prep = make_preprocessor()
        Xtr_prepared = prep.fit_transform(Xtr, ytr)

        if setting == "With chi-square selection":
            # Select up to 2,000 features, or all available features
            # if the dataset produces fewer than 2,000.
            k = min(2000, Xtr_prepared.shape[1])
            selector = SelectKBest(score_func=chi2, k=k)
            Xtr_model = selector.fit_transform(Xtr_prepared, ytr)
        else:
            selector = None
            Xtr_model = Xtr_prepared

        model = clone(classifier)
        model.fit(Xtr_model, ytr)

        Xte_prepared = prep.transform(Xte)
        Xte_model = (
            selector.transform(Xte_prepared)
            if selector is not None
            else Xte_prepared
        )

        predictions = model.predict(Xte_model)
        probabilities = model.predict_proba(Xte_model)[:, 1]

        steps = [("prep", prep)]
        if selector is not None:
            steps.append(("feature_selection", selector))
        steps.append(("classifier", model))
        pipeline = Pipeline(steps)

        scores = {
            "Setting": setting,
            "Model": model_name,
            "Accuracy": accuracy_score(yte, predictions),
            "Precision": precision_score(
                yte, predictions, zero_division=0
            ),
            "Recall": recall_score(
                yte, predictions, zero_division=0
            ),
            "F1": f1_score(yte, predictions, zero_division=0),
            "ROC-AUC": roc_auc_score(yte, probabilities),
        }

        rows.append(scores)
        fitted_models[(setting, model_name)] = pipeline

        print(
            f"{setting:28s} {model_name:20s} "
            f"F1={scores['F1']:.3f}"
        )

# ---------- 6. Show evaluation results ----------
results = pd.DataFrame(rows).round(4)
display(results)

results.pivot(
    index="Model",
    columns="Setting",
    values="F1",
).plot.bar(
    figsize=(10, 5),
    title="Model comparison by F1 score",
)
plt.ylabel("F1 score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

best = results.sort_values(
    ["F1", "ROC-AUC"],
    ascending=False,
).iloc[0]

best_key = (best["Setting"], best["Model"])
best_model = fitted_models[best_key]

print(
    f"\nBest model: {best['Model']} | {best['Setting']} | "
    f"F1={best['F1']} | Recall={best['Recall']}"
)
print(
    "Confusion matrix:\n",
    confusion_matrix(yte, best_model.predict(Xte)),
)

# ---------- 7. Global word explanation ----------
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=2,
)
Xtr_text = tfidf.fit_transform(Xtr["text"])

word_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
)
word_model.fit(Xtr_text, ytr)

word_weights = pd.Series(
    word_model.coef_[0],
    index=tfidf.get_feature_names_out(),
).sort_values()

top_words = pd.concat(
    [word_weights.head(15), word_weights.tail(15)]
)

bar_colors = ["tab:green"] * 15 + ["tab:red"] * 15

top_words.plot.barh(
    figsize=(8, 7),
    color=bar_colors,
    title="Word weights (green = genuine, red = fraudulent)",
)
plt.tight_layout()
plt.show()

# ---------- 8. Prediction and explanations ----------
def probability_fake(row_dict):
    row = add_features(pd.DataFrame([row_dict]))
    return float(best_model.predict_proba(row)[0, 1])


def explain_words(form, base_probability, top_n=6, max_words=60):
    combined_text = " ".join(
        str(form.get(column, ""))
        for column in TEXT_COLS
    )

    words = list(
        dict.fromkeys(
            re.findall(r"[A-Za-z]{4,}", combined_text.lower())
        )
    )[:max_words]

    changes = []

    for word in words:
        modified = {
            key: (
                re.sub(
                    rf"\b{re.escape(word)}\b",
                    " ",
                    str(value),
                    flags=re.I,
                )
                if key in TEXT_COLS
                else value
            )
            for key, value in form.items()
        }

        change = base_probability - probability_fake(modified)
        changes.append((word, change))

    changes.sort(key=lambda item: item[1])

    fake_words = [
        (word, round(change * 100, 2))
        for word, change in changes[::-1][:top_n]
        if change > 0
    ]

    genuine_words = [
        (word, round(change * 100, 2))
        for word, change in changes[:top_n]
        if change < 0
    ]

    return fake_words, genuine_words


# ---------- 9. High-contrast interface styling ----------
CSS = """
.gradio-container {
    background:
      radial-gradient(ellipse at top left, #dbeafe 0%, transparent 45%),
      linear-gradient(135deg, #eff6ff 0%, #f5f3ff 50%, #fdf2f8 100%);
    min-height: 100vh;
}

.fj-hero {
    background: linear-gradient(120deg, #1e3a8a, #6d28d9 60%, #db2777);
    color: #ffffff !important;
    padding: 26px 30px;
    border-radius: 16px;
    margin-bottom: 12px;
}

.fj-hero h1 {
    margin: 0 0 6px;
    font-size: 30px;
    color: #ffffff !important;
}

.fj-hero p {
    margin: 0;
    opacity: 1;
    font-size: 15px;
    color: #ffffff !important;
}

.fj-pills span {
    display: inline-block;
    background: rgba(255,255,255,.20);
    color: #ffffff !important;
    padding: 5px 12px;
    border-radius: 20px;
    margin: 10px 6px 0 0;
    font-size: 13px;
}

.fj-card {
    background: #ffffff !important;
    border: 1px solid #cbd5e1;
    border-radius: 14px;
    padding: 22px;
    color: #111827 !important;
}

.fj-card h3 {
    color: #111827 !important;
    font-size: 18px;
    font-weight: 700;
    margin-top: 22px;
}

.fj-card p,
.fj-card div,
.fj-card small,
.fj-card b {
    color: inherit;
}

.fj-top {
    display: flex;
    gap: 20px;
    align-items: center;
    flex-wrap: wrap;
}

.fj-gauge {
    width: 140px;
    height: 140px;
    flex: 0 0 140px;
    border-radius: 50%;
    display: grid;
    place-items: center;
    background: conic-gradient(
        var(--c) calc(var(--p) * 1%),
        #d1d5db 0
    );
}

.fj-gauge-inner {
    width: 108px;
    height: 108px;
    border-radius: 50%;
    background: #ffffff !important;
    color: #111827 !important;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
}

.fj-gauge-inner b {
    color: #111827 !important;
    font-size: 25px;
    font-weight: 800;
}

.fj-gauge-inner small {
    color: #374151 !important;
    font-size: 12px;
}

.fj-badge {
    display: inline-block;
    color: #ffffff !important;
    padding: 8px 14px;
    border-radius: 20px;
    font-weight: 700;
}

.fj-risk-text {
    color: #1f2937 !important;
    font-size: 15px;
    line-height: 1.6;
}

.fj-flag {
    background: #fff7ed !important;
    border-left: 5px solid #ea580c;
    padding: 10px 12px;
    margin: 7px 0;
    border-radius: 7px;
    font-size: 15px;
    color: #7c2d12 !important;
}

.fj-ok {
    background: #ecfdf5 !important;
    border-left: 5px solid #059669;
    padding: 10px 12px;
    border-radius: 7px;
    font-size: 15px;
    color: #065f46 !important;
}

.fj-chip {
    display: inline-block;
    padding: 6px 12px;
    margin: 4px 5px 4px 0;
    border-radius: 18px;
    font-size: 14px;
    font-weight: 600;
}

.fj-red {
    background: #fee2e2 !important;
    color: #991b1b !important;
}

.fj-green {
    background: #dcfce7 !important;
    color: #166534 !important;
}

.fj-tip {
    margin-top: 18px;
    font-size: 13px;
    line-height: 1.6;
    color: #374151 !important;
}

.fj-empty {
    color: #374151 !important;
}

.fj-foot {
    text-align: center;
    color: #374151 !important;
    font-size: 13px;
    margin-top: 8px;
}

.fj-foot span {
    display: inline-block;
    background: #e0e7ff !important;
    color: #312e81 !important;
    padding: 5px 10px;
    border-radius: 12px;
    margin: 3px;
}
"""

HERO = f"""
<div class="fj-hero">
  <h1>🛡️ Fake Job &amp; Internship Detector</h1>
  <p>Enter a job posting to check for possible scam patterns.</p>
  <div class="fj-pills">
    <span>🤖 {_html.escape(str(best['Model']))}</span>
    <span>🎯 F1: {best['F1']}</span>
    <span>🔎 Explainable AI</span>
  </div>
</div>
"""

PLACEHOLDER = """
<div class="fj-card" style="text-align:center;padding:40px 20px">
  <div style="font-size:54px;color:#111827">🕵️</div>
  <h3 style="color:#111827">Waiting for a posting</h3>
  <p class="fj-empty">
    Fill in the form, then press Analyze posting.
  </p>
</div>
"""


def render_result(probability, fake_words, genuine_words, flags):
    percentage = round(probability * 100, 1)

    if probability >= 0.5:
        color = "#dc2626"
        verdict = "⚠️ FAKE / SUSPICIOUS"
        risk = "High risk"
    elif probability >= 0.3:
        color = "#d97706"
        verdict = "🤔 NEEDS CAUTION"
        risk = "Medium risk"
    else:
        color = "#059669"
        verdict = "✅ LIKELY GENUINE"
        risk = "Low risk"

    flags_html = "".join(
        f'<div class="fj-flag">🚩 {_html.escape(flag)}</div>'
        for flag in flags
    )

    if not flags_html:
        flags_html = (
            '<div class="fj-ok">'
            "✔ No rule-based red flags found"
            "</div>"
        )

    fake_html = "".join(
        f'<span class="fj-chip fj-red">'
        f'{_html.escape(word)} +{change}%</span>'
        for word, change in fake_words
    )

    if not fake_html:
        fake_html = (
            '<span class="fj-empty">No words identified</span>'
        )

    genuine_html = "".join(
        f'<span class="fj-chip fj-green">'
        f'{_html.escape(word)} {change}%</span>'
        for word, change in genuine_words
    )

    if not genuine_html:
        genuine_html = (
            '<span class="fj-empty">No words identified</span>'
        )

    return f"""
    <div class="fj-card">
      <div class="fj-top">
        <div class="fj-gauge" style="--p:{percentage};--c:{color}">
          <div class="fj-gauge-inner">
            <b>{percentage}%</b>
            <small>fake probability</small>
          </div>
        </div>

        <div>
          <span class="fj-badge" style="background:{color}">
            {verdict}
          </span>
          <p class="fj-risk-text">
            {risk}. Verify the company through its official website
            before sharing documents or paying fees.
          </p>
        </div>
      </div>

      <h3>🚩 Red flags</h3>
      {flags_html}

      <h3>🔴 Words that increased the fake score</h3>
      {fake_html}

      <h3>🟢 Words that lowered the fake score</h3>
      {genuine_html}

      <div class="fj-tip">
        The word explanation removes individual words and measures how
        the model's fake probability changes.
      </div>
    </div>
    """


def analyze(
    title,
    location,
    salary,
    industry,
    function,
    employment_type,
    experience,
    education,
    profile,
    description,
    requirements,
    benefits,
    remote,
    logo,
    questions,
):
    form = {
        "title": title or "",
        "location": location or "",
        "salary_range": salary or "",
        "industry": industry or "",
        "function": function or "",
        "employment_type": employment_type or "",
        "required_experience": experience or "",
        "required_education": education or "",
        "company_profile": profile or "",
        "description": description or "",
        "requirements": requirements or "",
        "benefits": benefits or "",
        "telecommuting": int(bool(remote)),
        "has_company_logo": int(bool(logo)),
        "has_questions": int(bool(questions)),
    }

    if not form["title"].strip() and not form["description"].strip():
        return PLACEHOLDER

    probability = probability_fake(form)
    fake_words, genuine_words = explain_words(form, probability)

    all_text = " ".join(str(value) for value in form.values())
    flags = []

    if not form["company_profile"].strip():
        flags.append("No company profile was provided")

    if not form["has_company_logo"]:
        flags.append("No company logo was indicated")

    if re.search(r"@(gmail|yahoo|outlook|hotmail)\.", all_text, re.I):
        flags.append("A free email address appears in the posting")

    if re.search(
        r"registration fee|deposit|western union|whatsapp|pay ",
        all_text,
        re.I,
    ):
        flags.append("The posting mentions payment or messaging apps")

    if re.search(
        r"no experience|urgent|earn \w+ (daily|weekly)|guaranteed",
        all_text,
        re.I,
    ):
        flags.append(
            "The posting uses urgent or unusually attractive wording"
        )

    return render_result(
        probability,
        fake_words,
        genuine_words,
        flags,
    )


# ---------- 10. Build the interface ----------
EMPLOYMENT_TYPES = [
    "",
    "Full-time",
    "Part-time",
    "Contract",
    "Temporary",
    "Other",
]

EXPERIENCE_LEVELS = [
    "",
    "Internship",
    "Entry level",
    "Associate",
    "Mid-Senior level",
    "Director",
    "Executive",
    "Not Applicable",
]

EDUCATION_LEVELS = [
    "",
    "Bachelor's Degree",
    "Master's Degree",
    "High School or equivalent",
    "Certification",
    "Unspecified",
]

with gr.Blocks(
    title="Fake Job & Internship Detector",
    css=CSS,
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="violet",
        neutral_hue="slate",
    ),
) as demo:
    gr.HTML(HERO)

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### 📝 Job details")

            with gr.Row():
                title_input = gr.Textbox(
                    label="Job title",
                    placeholder="e.g. Python Developer Intern",
                )
                location_input = gr.Textbox(
                    label="Location",
                    placeholder="e.g. IN, Hyderabad",
                )

            with gr.Row():
                salary_input = gr.Textbox(
                    label="Salary / stipend",
                    placeholder="e.g. 10000-15000 per month",
                )
                industry_input = gr.Textbox(
                    label="Industry",
                    placeholder="Information Technology",
                )
                function_input = gr.Textbox(
                    label="Function",
                    placeholder="Engineering",
                )

            with gr.Row():
                employment_input = gr.Dropdown(
                    choices=EMPLOYMENT_TYPES,
                    value="",
                    label="Employment type",
                )
                experience_input = gr.Dropdown(
                    choices=EXPERIENCE_LEVELS,
                    value="",
                    label="Experience level",
                )
                education_input = gr.Dropdown(
                    choices=EDUCATION_LEVELS,
                    value="",
                    label="Education",
                )

            profile_input = gr.Textbox(
                label="Company profile",
                lines=2,
                placeholder="About the company...",
            )

            description_input = gr.Textbox(
                label="Job description",
                lines=5,
                placeholder="Paste the job posting here...",
            )

            with gr.Row():
                requirements_input = gr.Textbox(
                    label="Requirements",
                    lines=2,
                )
                benefits_input = gr.Textbox(
                    label="Benefits",
                    lines=2,
                )

            with gr.Row():
                remote_input = gr.Checkbox(label="🏠 Remote job")
                logo_input = gr.Checkbox(
                    label="🏢 Company logo present",
                    value=True,
                )
                questions_input = gr.Checkbox(
                    label="❓ Screening questions",
                )

            with gr.Row():
                analyze_button = gr.Button(
                    "🔍 Analyze posting",
                    variant="primary",
                )
                clear_button = gr.ClearButton(value="Clear")

        with gr.Column(scale=4):
            output = gr.HTML(value=PLACEHOLDER)

    input_components = [
        title_input,
        location_input,
        salary_input,
        industry_input,
        function_input,
        employment_input,
        experience_input,
        education_input,
        profile_input,
        description_input,
        requirements_input,
        benefits_input,
        remote_input,
        logo_input,
        questions_input,
    ]

    analyze_button.click(
        fn=analyze,
        inputs=input_components,
        outputs=output,
    )

    clear_button.add(input_components + [output])

    gr.HTML(
        """
        <div class="fj-foot">
          <span>🎯 SDG 8: Decent Work &amp; Economic Growth</span>
          <span>📚 SDG 4: Quality Education</span>
          <span>⚖️ SDG 16: Strong Institutions</span>
        </div>
        """
    )

# share=True creates a public Gradio link.
# Use demo.launch() if you do not need a public link.
demo.launch(share=True)

### 🚀 Converting the Project to Streamlit and Pushing to GitHub
Now we will convert our interactive web app from Gradio to Streamlit and save it to `app.py`, which is standard for Streamlit deployments on platforms like Streamlit Community Cloud.

## Streamlit app

The deployable Streamlit implementation is maintained in [`app.py`](app.py). Run it with `streamlit run app.py`; dependencies are listed in `requirements.txt`.


In [ ]:
import pickle

# Save the best model and our column groups so the Streamlit app can use them exactly as defined in the notebook
model_assets = {
    'model': best_model,
    'TEXT_COLS': TEXT_COLS,
    'CAT_COLS': CAT_COLS,
    'BINARY_COLS': BINARY_COLS,
    'NUM_COLS': NUM_COLS,
    'ALL_INPUTS': ALL_INPUTS
}

with open('best_model.pkl', 'wb') as f:
    pickle.dump(model_assets, f)

print("Successfully saved trained model pipeline and column configurations to best_model.pkl!")

## Streamlit app

The deployable Streamlit implementation is maintained in [`app.py`](app.py). Run it with `streamlit run app.py`; dependencies are listed in `requirements.txt`.


### 🛠️ Step 2: Push to GitHub
Run this cell to set up git, commit your files, and push to your remote GitHub repository.

## GitHub and credentials

Do not put a GitHub token in a notebook or remote URL. Commit and push from a trusted local Git client using GitHub authentication, or use the GitHub web interface. The deployable app is `app.py` in this repository.


## Deploy with Streamlit Community Cloud

Push this repository to GitHub, create a Streamlit Community Cloud app from the `main` branch, set the app file to `app.py`, and deploy. Install dependencies from `requirements.txt`. Upload a labeled CSV in the app sidebar to train the model. See `README.md` for local setup and deployment details.
